# AMEX Enterprise Credit Risk Platform
## Notebook 28 — Phase 2, Problem 4: Delinquency Escalation / Loss Severity — Statistical Validation & Deployment
### CRISP-DM: Evaluation + Deployment

Third of 4 notebooks for Problem 4. Statistically stress-tests Notebook 27's real severity tiers (chi-square
independence test, a two-proportion z-test between the extreme tiers, bootstrap confidence intervals on each
tier's default rate, and a split-half Population Stability Index), then packages the frozen scoring logic into
a standalone, runnable `severity_scorer.py` module + a `severity_scoring_bundle.json` — and proves the two agree
with Notebook 27's own output on a real holdout customer before calling it deployment-ready.

**Honesty boundary (unchanged):** every statistic, p-value, confidence interval, and self-test result below is
computed live from real data this run. Only the dollar LGD per tier (loaded from Notebook 26) stays an
`ASSUMPTION`.

**Deliverables:** `p4_deployment_readiness_checklist.csv`, `severity_scoring_bundle.json`, `severity_scorer.py`,
an inline tier-default-rate-with-CI chart.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD PROBLEM 1's REAL FEATURES + NOTEBOOK 26/27 OUTPUTS
# =============================================================================
import os
import sys
import json
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Problem 1's Real Features + Notebook 26/27 Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ROOT = PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
P1_ARTIFACTS = P1_ROOT / "artifacts"
# Legacy pre-rename data-cache folder -- the large engineered/raw CSVs were deliberately excluded
# from the Phase 1 folder reorg copy and were confirmed still living under this old folder name.
P1_LEGACY_ROOT = P1_ROOT.parent / "Problem 1 Credit Default_Probability of Default"
P4_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "04_Problem4_Delinquency_Escalation_Loss_Severity"
ARTIFACTS_DIR = P4_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PILLAR_DIRS = {
    "p4_policy": P4_ROOT / "01_LGD_Policy",
    "p4_modeling": P4_ROOT / "02_LGD_Modeling",
    "p4_validation_deployment": P4_ROOT / "03_Validation_Deployment",
    "p4_reporting_packaging": P4_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = P1_ARTIFACTS / "project_config.json"
NB04_SUMMARY_PATH = P1_ARTIFACTS / "notebook_04_summary.json"
LGD_POLICY_PATH = PILLAR_DIRS["p4_policy"] / "lgd_policy.json"
WEIGHTS_PATH = PILLAR_DIRS["p4_modeling"] / "severity_feature_weights.csv"
SCORES_PATH = PILLAR_DIRS["p4_modeling"] / "severity_scores_holdout.csv"
TIER_VALIDATION_PATH = PILLAR_DIRS["p4_modeling"] / "tier_validation_summary.json"
for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (NB04_SUMMARY_PATH, "run Problem 1's Notebook 04 first."),
    (LGD_POLICY_PATH, "run Notebook 26 (this problem's Notebook 1) first."),
    (WEIGHTS_PATH, "run Notebook 27 (this problem's Notebook 2) first."),
    (SCORES_PATH, "run Notebook 27 (this problem's Notebook 2) first."),
    (TIER_VALIDATION_PATH, "run Notebook 27 (this problem's Notebook 2) first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected project folder.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)
with open(LGD_POLICY_PATH, "r", encoding="utf-8") as f:
    LGD_POLICY = json.load(f)
with open(TIER_VALIDATION_PATH, "r", encoding="utf-8") as f:
    TIER_VALIDATION = json.load(f)

RANDOM_SEED = P1_CONFIG["random_seed"]
TIER_ORDER = LGD_POLICY["tier_order"]
LGD_BY_TIER = {v["tier"]: v["lgd"] for v in LGD_POLICY["lgd_by_tier"]["values"]}
CUT_LOW = TIER_VALIDATION["cutpoints"]["low_upper_bound"]
CUT_HIGH = TIER_VALIDATION["cutpoints"]["moderate_upper_bound"]


def _resolve_engineered_file(filename: str) -> Path:
    """Same 3-candidate resolver as Notebook 27 (deterministic new-structure path, legacy
    pre-rename folder, summary-JSON path as last resort) -- a candidate must be real data
    (>10KB) to count, ruling out a stray placeholder/note file of the same name."""
    _candidates = [
        P1_ROOT / "Feature_Engineering" / filename,
        P1_LEGACY_ROOT / "Feature_Engineering" / filename,
        Path(NB04_SUMMARY["output_files"][filename]),
    ]
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > 10_000:
            return _c
    raise FileNotFoundError(
        f"{filename} not found (as real data, >10KB) at any checked location:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: re-run Problem 1's Notebook 04 from its current Phase1_Foundation location."
    )


TEST_SPLIT_ENG_PATH = _resolve_engineered_file("test_split_engineered.csv")
# Note: train_split_engineered.csv is NOT re-resolved/reloaded here on purpose -- Section 7 below
# reuses Notebook 27's own real mean/std/weight values (now saved in severity_feature_weights.csv)
# instead of recomputing them from a fresh reload of the huge train file. Recomputing independently
# was tried and rejected: see Section 7's comment for why.

print(f"Severity tiers (Notebook 26, real)     : {TIER_ORDER}")
print(f"LGD by tier (Notebook 26, ASSUMPTION)   : {LGD_BY_TIER}")
print(f"Frozen cutpoints (Notebook 27, real)    : low<={CUT_LOW:.4f} < moderate<={CUT_HIGH:.4f}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    from scipy.stats import chi2_contingency, norm
except ImportError:
    missing.append("scipy")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

np.random.seed(RANDOM_SEED)
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD NOTEBOOK 27's REAL OUTPUTS
# =============================================================================
_section("SECTION 3: Load Notebook 27's Real Outputs")

scores_df = pd.read_csv(SCORES_PATH)
weights_df = pd.read_csv(WEIGHTS_PATH)
scores_df["severity_tier"] = pd.Categorical(scores_df["severity_tier"], categories=TIER_ORDER, ordered=True)
print(f"Holdout customers loaded (real, from Notebook 27): {len(scores_df):,}")
print(scores_df["severity_tier"].value_counts().reindex(TIER_ORDER).to_string())
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: STATISTICAL VALIDATION -- CHI-SQUARE INDEPENDENCE + TWO-PROPORTION Z-TEST
# =============================================================================
_section("SECTION 4: Statistical Validation -- Chi-Square Independence + Two-Proportion Z-Test")

_contingency = pd.crosstab(scores_df["severity_tier"], scores_df["target"])
_chi2, _chi2_p, _dof, _ = chi2_contingency(_contingency)
_n = len(scores_df)
_cramers_v = float(np.sqrt(_chi2 / (_n * (min(_contingency.shape) - 1))))
print(f"Chi-square test (tier independent of default?): chi2={_chi2:.2f}, dof={_dof}, p-value={_chi2_p:.2e}")
print(f"Cram\u00e9r's V (effect size)                      : {_cramers_v:.4f}")

_low_row = _contingency.loc[TIER_ORDER[0]]
_severe_row = _contingency.loc[TIER_ORDER[-1]]
_n_low, _x_low = int(_low_row.sum()), int(_low_row.get(1, 0))
_n_severe, _x_severe = int(_severe_row.sum()), int(_severe_row.get(1, 0))
_p_low, _p_severe = _x_low / _n_low, _x_severe / _n_severe
_p_pool = (_x_low + _x_severe) / (_n_low + _n_severe)
_se = np.sqrt(_p_pool * (1 - _p_pool) * (1 / _n_low + 1 / _n_severe))
_z = (_p_severe - _p_low) / _se if _se > 0 else float("inf")
_z_p = float(2 * (1 - norm.cdf(abs(_z))))
print(f"Two-proportion z-test (Severe vs Low default rate): z={_z:.2f}, p-value={_z_p:.2e} "
      f"(Low={_p_low:.2%}, Severe={_p_severe:.2%})")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BOOTSTRAP CONFIDENCE INTERVALS ON PER-TIER DEFAULT RATE
# =============================================================================
_section("SECTION 5: Bootstrap Confidence Intervals on Per-Tier Default Rate")

N_BOOT = 1000
_rng = np.random.default_rng(RANDOM_SEED)
_bootstrap_rows = []
for _t in TIER_ORDER:
    _y = scores_df.loc[scores_df["severity_tier"] == _t, "target"].to_numpy()
    _boot_means = np.array([_rng.choice(_y, size=len(_y), replace=True).mean() for _ in range(N_BOOT)])
    _lo, _hi = np.percentile(_boot_means, [2.5, 97.5])
    _bootstrap_rows.append({"tier": _t, "n": len(_y), "observed_default_rate": float(_y.mean()),
                             "ci_95_low": float(_lo), "ci_95_high": float(_hi)})
bootstrap_df = pd.DataFrame(_bootstrap_rows)
print(bootstrap_df.to_string(index=False))
print(f"\n(N_BOOT={N_BOOT} resamples per tier, random_seed={RANDOM_SEED} -- reproducible.)")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: SPLIT-HALF POPULATION STABILITY INDEX (PSI)
# =============================================================================
_section("SECTION 6: Split-Half Population Stability Index (PSI)")

_shuffled = scores_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
_half_a, _half_b = _shuffled.iloc[: len(_shuffled) // 2], _shuffled.iloc[len(_shuffled) // 2:]
_dist_a = _half_a["severity_tier"].value_counts(normalize=True).reindex(TIER_ORDER).fillna(0.0)
_dist_b = _half_b["severity_tier"].value_counts(normalize=True).reindex(TIER_ORDER).fillna(0.0)
_eps = 1e-6
PSI = float(sum((_dist_b[_t] - _dist_a[_t]) * np.log((_dist_b[_t] + _eps) / (_dist_a[_t] + _eps))
                 for _t in TIER_ORDER))
# Standard industry PSI convention (not this-dataset-specific): <0.10 stable, 0.10-0.25 moderate
# shift worth monitoring, >0.25 significant shift -- interpretation only, computed value is real.
_psi_verdict = "stable" if PSI < 0.10 else ("moderate shift" if PSI < 0.25 else "significant shift")
print(f"Split-half PSI on severity tier distribution: {PSI:.4f} ({_psi_verdict})")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: DEPLOYMENT BUNDLE -- REFREEZE STANDARDIZATION STATS ON TRAIN SPLIT
# =============================================================================
_section("SECTION 7: Deployment Bundle -- Carry Forward Notebook 27's Real Standardization Stats")

_bundle_features = weights_df["feature"].tolist()

# --- Earlier version of this notebook RE-COMPUTED mean/std by re-loading train_split_engineered.csv
#     and calling np.nanmean/np.nanstd again here. That is a second, independent computation of a
#     value Notebook 27 already computed once for real -- and on the real dataset it measurably
#     disagreed with Notebook 27's own in-memory values by a small but non-negligible amount (~1e-4
#     in the final score, confirmed on real data), enough to flip a handful of customers whose true
#     score sits close to a tier cutpoint. Separately, the weight column this file used to load was
#     ALSO rounded to 5 decimal places for CSV readability, which independently contributed real
#     error when summed across 200+ features. Both are now fixed at the source: Notebook 27 saves
#     full-precision mean/std/normalized_weight directly in severity_feature_weights.csv, and this
#     notebook simply reuses those exact real values -- no second computation, so no way for the two
#     to disagree. This also means Notebook 28 no longer needs to touch the multi-GB train file. ---
_missing_cols = [c for c in ("mean", "std") if c not in weights_df.columns]
if _missing_cols:
    raise RuntimeError(
        f"severity_feature_weights.csv is missing column(s) {_missing_cols} -- it was generated by an "
        "older version of Notebook 27 that didn't save full-precision mean/std alongside the weights.\n"
        "Fix: re-run Notebook 27 (this problem's Notebook 2) once to regenerate it, then re-run this "
        "notebook."
    )

FEATURE_MEAN = dict(zip(weights_df["feature"], weights_df["mean"].astype(float)))
FEATURE_STD = dict(zip(weights_df["feature"], weights_df["std"].astype(float)))
FEATURE_WEIGHT = dict(zip(weights_df["feature"], weights_df["normalized_weight"].astype(float)))
FEATURE_DIRECTION = dict(zip(weights_df["feature"], weights_df["direction"]))

SCORING_BUNDLE = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "features": _bundle_features,
    "weights": FEATURE_WEIGHT, "directions": FEATURE_DIRECTION,
    "means": FEATURE_MEAN, "stds": FEATURE_STD,
    "cut_low": CUT_LOW, "cut_high": CUT_HIGH, "tier_order": TIER_ORDER,
    "lgd_by_tier": LGD_BY_TIER,
}
bundle_path = PILLAR_DIRS["p4_validation_deployment"] / "severity_scoring_bundle.json"
with open(bundle_path, "w", encoding="utf-8") as f:
    json.dump(SCORING_BUNDLE, f, indent=2)
print(f"Bundle covers {len(_bundle_features)} features, carried forward at full precision from "
      f"Notebook 27's own real computation (no second computation, so no way to disagree).")
print(f"\u2705 Saved -> {bundle_path.name} (this problem's 03_Validation_Deployment folder)")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: GENERATE severity_scorer.py -- STANDALONE RUNNABLE SCORING MODULE
# =============================================================================
_section("SECTION 8: Generate severity_scorer.py -- Standalone Runnable Scoring Module")

_scorer_source = """# Standalone Escalation Severity scorer for Problem 4 (Delinquency Escalation / Loss
# Severity). Generated by Notebook 28 -- loads severity_scoring_bundle.json (saved alongside this
# file) and scores one customer's real D_* engineered features into a severity tier + LGD.
# No training happens here; every weight/cutpoint/mean/std is frozen from Notebook 27's real run.
import json
import math
from pathlib import Path

BUNDLE_PATH = Path(__file__).parent / "severity_scoring_bundle.json"


def load_bundle(path=BUNDLE_PATH):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _is_missing(value):
    # None covers a plain missing key/explicit None; NaN (float('nan')) is how pandas/numpy
    # represent a missing statement value once loaded -- "value != value" is only ever True for
    # NaN under IEEE754, so this catches both without importing pandas into this standalone file.
    return value is None or (isinstance(value, float) and math.isnan(value))


def score_customer(feature_dict, bundle):
    # feature_dict: {feature_name: value} for at least the bundle's feature list.
    # Returns {"severity_score": float, "severity_tier": str, "lgd": float}.
    # Iterates in a fixed alphabetical order, NOT bundle["features"]'s stored order -- floating
    # point addition is not associative, so summing the same terms in a different order can shift
    # the last bit or two of the result. That is invisible almost everywhere, but a score that
    # lands extremely close to a cutpoint can round to the other side of it. Notebook 27's own
    # pipeline always sums in alphabetical feature-name order, so this must match that exactly.
    score = 0.0
    for feat in sorted(bundle["features"]):
        raw = feature_dict.get(feat)
        if _is_missing(raw):
            raw = bundle["means"][feat]  # same missing-value handling as Notebook 27
        z = (float(raw) - bundle["means"][feat]) / bundle["stds"][feat]
        score += bundle["weights"][feat] * bundle["directions"][feat] * z

    if score <= bundle["cut_low"]:
        tier = bundle["tier_order"][0]
    elif score <= bundle["cut_high"]:
        tier = bundle["tier_order"][1]
    else:
        tier = bundle["tier_order"][2]

    return {"severity_score": score, "severity_tier": tier, "lgd": bundle["lgd_by_tier"][tier]}


if __name__ == "__main__":
    _bundle = load_bundle()
    _example = {f: _bundle["means"][f] for f in _bundle["features"]}
    print(score_customer(_example, _bundle))
"""
scorer_path = PILLAR_DIRS["p4_validation_deployment"] / "severity_scorer.py"
with open(scorer_path, "w", encoding="utf-8") as f:
    f.write(_scorer_source)
print(f"\u2705 Saved -> {scorer_path.name} (this problem's 03_Validation_Deployment folder)")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: LIVE SELF-TEST -- IMPORT severity_scorer.py, SCORE EVERY REAL HOLDOUT CUSTOMER
# =============================================================================
_section("SECTION 9: Live Self-Test -- Import severity_scorer.py, Score Every Real Holdout Customer")

# --- Testing a single customer is not enough: a summation-order or edge-case bug typically only
#     flips the outcome for customers whose score lands extremely close to a cutpoint, so a
#     one-customer check can pass by luck while others silently mismatch. Every real holdout
#     customer is checked below instead. ---
_spec = importlib.util.spec_from_file_location("severity_scorer", str(scorer_path))
severity_scorer = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(severity_scorer)

_holdout_full = pl.read_csv(
    str(TEST_SPLIT_ENG_PATH), columns=["customer_ID"] + _bundle_features
).to_pandas()
_check_df = scores_df[["customer_ID", "severity_score", "severity_tier", "lgd_assigned"]].merge(
    _holdout_full, on="customer_ID", how="left", validate="one_to_one")
if _check_df[_bundle_features].isnull().all(axis=1).any():
    raise RuntimeError("One or more holdout customers had no matching feature row on reload. "
                        "Fix: re-run Notebook 27, then this notebook.")

# --- A tier/LGD mismatch is classified two ways:
#     (1) a BOUNDARY TIE -- the standalone scorer's recomputed severity_score differs from the
#         pipeline's own real saved score by less than BOUNDARY_EPS. This can only happen when the
#         pipeline's true score already sat within BOUNDARY_EPS of a tier cutpoint, because a tier
#         only changes when a score crosses CUT_LOW/CUT_HIGH -- so a sub-epsilon score difference
#         flipping the tier PROVES it was a coin-flip-close boundary case, not a logic error. This is
#         an inherent, provable consequence of floating-point non-associativity: Notebook 27 sums each
#         customer's weighted terms via a numpy-vectorized loop, the standalone scorer sums the same
#         terms (same values, same order) via scalar Python arithmetic -- even identical mathematical
#         order of operations is not guaranteed to produce bit-identical results across the two
#         execution paths, so a last-few-bits difference is expected, not a defect.
#     (2) a HARD MISMATCH -- score difference is NOT tiny, meaning something is actually wrong
#         (bad feature list, wrong weights/means/stds, a real logic bug). This must be zero. ---
BOUNDARY_EPS = 1e-9
_hard_mismatches, _boundary_ties = [], []
for _rec in _check_df.to_dict("records"):
    _feat_dict = {f: _rec[f] for f in _bundle_features}
    _res = severity_scorer.score_customer(_feat_dict, SCORING_BUNDLE)
    _tier_or_lgd_differs = (_res["severity_tier"] != str(_rec["severity_tier"])) or \
        (abs(_res["lgd"] - float(_rec["lgd_assigned"])) > 1e-9)
    if not _tier_or_lgd_differs:
        continue
    _score_diff = abs(_res["severity_score"] - float(_rec["severity_score"]))
    _row = {"customer_ID": _rec["customer_ID"], "pipeline_tier": _rec["severity_tier"],
            "scorer_tier": _res["severity_tier"], "pipeline_lgd": _rec["lgd_assigned"],
            "scorer_lgd": _res["lgd"], "score_diff": _score_diff}
    if _score_diff < BOUNDARY_EPS:
        _boundary_ties.append(_row)
    else:
        _hard_mismatches.append(_row)

_n_checked = len(_check_df)
_n_hard = len(_hard_mismatches)
_n_boundary = len(_boundary_ties)
_boundary_rate = _n_boundary / _n_checked if _n_checked else 0.0
# Sanity ceiling: if "boundary ties" ever cover more than 0.1% of customers, that pattern looks more
# like a real systematic bug hiding behind this classification than genuine floating-point noise --
# treat that as a hard failure too rather than trusting the epsilon check blindly.
_boundary_rate_sane = _boundary_rate <= 0.001
_self_test_passed = (_n_hard == 0) and _boundary_rate_sane

print(f"Holdout customers checked                : {_n_checked:,}")
print(f"Hard mismatches (real defect, must be 0)  : {_n_hard:,}")
print(f"Boundary ties (score differs by <{BOUNDARY_EPS:g}, expected float noise): {_n_boundary:,} "
      f"({_boundary_rate:.4%} of customers)")
if _hard_mismatches:
    print("\nHard mismatches (investigate -- NOT explained by floating-point noise):")
    print(pd.DataFrame(_hard_mismatches).head(10).to_string(index=False))
if _boundary_ties:
    print("\nBoundary ties (informational only -- these customers' real scores sit within "
          f"{BOUNDARY_EPS:g} of a tier cutpoint, so the two independently-computed arithmetic paths "
          "can legitimately land on opposite sides of it):")
    print(pd.DataFrame(_boundary_ties).head(10).to_string(index=False))
if not _boundary_rate_sane:
    print(f"\n\u26a0\ufe0f Boundary-tie rate {_boundary_rate:.4%} exceeds the 0.10% sanity ceiling -- "
          "too high to trust as floating-point noise alone; treating as a hard failure.")
print(f"\nSelf-test {'PASSED' if _self_test_passed else 'FAILED'} -- standalone scorer "
      f"{'matches' if _self_test_passed else 'does NOT match'} the Notebook 27 pipeline on "
      f"{_n_checked - _n_hard:,}/{_n_checked:,} real holdout customers "
      f"({_n_boundary:,} of those as expected floating-point boundary ties, {_n_hard:,} as real mismatches).")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: DEPLOYMENT READINESS CHECKLIST
# =============================================================================
_section("SECTION 10: Deployment Readiness Checklist")

_checks_passed = True


def _check(label, condition, detail="", hard=True):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        if hard:
            _checks_passed = False
        _mark = "\u274c" if hard else "\u26a0\ufe0f"
        print(f"{_mark} {label}  {detail}")


_check(f"Standalone scorer matches Notebook 27's own output, allowing only float-boundary ties "
       f"(hard requirement) -- {_n_hard} real mismatches, {_n_boundary} boundary ties of {_n_checked:,}",
       _self_test_passed, hard=True)
_check("Tier is statistically associated with default (chi-square p < 0.05)", _chi2_p < 0.05, hard=True)
_check("Severe vs Low default-rate difference is statistically significant (z-test p < 0.05)",
       _z_p < 0.05, hard=True)
_check("Split-half PSI indicates a stable tiering scheme (PSI < 0.10)", PSI < 0.10,
       f"(measured={PSI:.4f})", hard=False)

readiness_rows = [
    {"check": "Standalone scorer matches pipeline output", "result": "PASS" if _self_test_passed else "FAIL"},
    {"check": "  - real (hard) mismatches", "result": f"{_n_hard:,} of {_n_checked:,}"},
    {"check": "  - float-boundary ties (expected, not a defect)", "result": f"{_n_boundary:,} ({_boundary_rate:.4%})"},
    {"check": "Chi-square independence test", "result": f"p={_chi2_p:.2e}"},
    {"check": "Two-proportion z-test (Severe vs Low)", "result": f"p={_z_p:.2e}"},
    {"check": "Split-half PSI", "result": f"{PSI:.4f} ({_psi_verdict})"},
    {"check": "Cram\u00e9r's V effect size", "result": f"{_cramers_v:.4f}"},
]
readiness_df = pd.DataFrame(readiness_rows)
readiness_path = PILLAR_DIRS["p4_validation_deployment"] / "p4_deployment_readiness_checklist.csv"
readiness_df.to_csv(readiness_path, index=False)
print(readiness_df.to_string(index=False))

if not _checks_passed:
    raise RuntimeError("One or more hard deployment-readiness checks failed. See \u274c line above.")
print(f"\n\u2705 Saved -> {readiness_path.name} (this problem's 03_Validation_Deployment folder)")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: INLINE CHART -- TIER DEFAULT RATE WITH 95% BOOTSTRAP CI
# =============================================================================
_section("SECTION 11: Inline Chart -- Tier Default Rate With 95% Bootstrap CI")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "surface": "#FFFFFF"}
fig, ax = plt.subplots(figsize=(7.5, 5), dpi=150)
_rates = bootstrap_df["observed_default_rate"].to_numpy()
_err_low = _rates - bootstrap_df["ci_95_low"].to_numpy()
_err_high = bootstrap_df["ci_95_high"].to_numpy() - _rates
ax.bar(TIER_ORDER, _rates, color=[VIZ["muted"], VIZ["accent"], VIZ["ink"]],
       yerr=[_err_low, _err_high], capsize=6)
ax.set_ylabel("Real observed default rate (95% bootstrap CI)")
ax.set_title("Problem 4: Escalation Severity Tier -- Default Rate With Confidence Interval")
fig.tight_layout()
chart_path = PILLAR_DIRS["p4_validation_deployment"] / "tier_default_rate_ci_chart.png"
fig.savefig(chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart_path.name} (this problem's 03_Validation_Deployment folder)")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 28 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 12: Write Notebook 28 Summary Artifact")

_expected_files = [bundle_path, scorer_path, readiness_path, chart_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)
if not _checks_passed:
    raise RuntimeError("One or more Notebook 28 output files failed to save. See \u274c line above.")

notebook_28_summary = {
    "notebook": "28_validation_deployment", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 4, "problem_name": "Delinquency Escalation / Loss Severity",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "chi_square_p_value": float(_chi2_p), "cramers_v": _cramers_v,
    "z_test_p_value": _z_p, "psi": PSI, "psi_verdict": _psi_verdict,
    "self_test_passed": bool(_self_test_passed),
    "self_test_customers_checked": _n_checked,
    "self_test_hard_mismatches": _n_hard,
    "self_test_boundary_ties": _n_boundary,
    "self_test_boundary_tie_rate": _boundary_rate,
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb28_summary_path = ARTIFACTS_DIR / "notebook_28_summary.json"
with open(nb28_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_28_summary, f, indent=2)
print(f"\u2705 Saved -> {nb28_summary_path.name} (this problem's artifacts folder)")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 13: Notebook 28 Complete -- Handoff to Notebook 29")

print("NOTEBOOK 28: STATISTICAL VALIDATION & DEPLOYMENT -- COMPLETE")
print(f"  Chi-square p-value (tier vs default)  : {_chi2_p:.2e}")
print(f"  Two-proportion z-test p-value          : {_z_p:.2e}")
print(f"  Split-half PSI                         : {PSI:.4f} ({_psi_verdict})")
print(f"  Standalone scorer self-test             : {'PASSED' if _self_test_passed else 'FAILED'}")
print(f"  Files produced                         : {len(_expected_files) + 1}")
for _p in _expected_files + [nb28_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                          : 29_financial_impact_reporting_packaging.ipynb")
print("\n\u2705 Ready to proceed.")
